In [ ]:
import os
import pandas as pd
import numpy as np

# Define directories
PROCESSED_DIR = os.path.join("..", "data", "processed")
REPORTS_DIR = os.path.join("..", "reports")

In [ ]:
# Load model predictions, sentiment scores, and portfolio weights
preds_df = pd.read_csv(os.path.join(REPORTS_DIR, "master_predictions.csv"))  # or model_predictions.csv
sentiment_df = pd.read_csv(os.path.join(PROCESSED_DIR, "sentiment_features.csv"))
weights_df = pd.read_csv(os.path.join(REPORTS_DIR, "portfolio_weights.csv"))

In [ ]:
# Aggregate average sentiment score per ticker
sentiment_summary = sentiment_df.groupby("Ticker")["Sentiment_Score"].mean().reset_index()

# Merge return predictions, sentiment, and MPT weights on Ticker
fused_df = preds_df.merge(sentiment_summary, on="Ticker", how="inner")
fused_df = fused_df.merge(weights_df, on="Ticker", how="inner")

# Display initial combined dataframe
fused_df.head()

In [ ]:
def generate_recommendation(row):
    exp_return = row["Expected Return (%)"]
    sentiment = row["Sentiment_Score"]
    weight = row["Weight"]
    
    # Fusion Logic Rules
    if exp_return > 1.0 and sentiment > 0.05 and weight > 0.05:
        return "STRONG BUY"
    elif exp_return > 0 and weight > 0:
        return "BUY"
    elif exp_return < -1.0 or sentiment < -0.1:
        return "SELL"
    else:
        return "HOLD"

# Apply decision rules
fused_df["Recommendation Signal"] = fused_df.apply(generate_recommendation, axis=1)
fused_df[["Ticker", "Expected Return (%)", "Sentiment_Score", "Weight", "Recommendation Signal"]]

In [ ]:
output_path = os.path.join(REPORTS_DIR, "recommendations.csv")
fused_df.to_csv(output_path, index=False)
print(f"Successfully generated trade recommendations and saved to {output_path}")